In [1]:
import pandas as pd
import numpy as np
import nbformat
import re
import tokenize
from pygments import lex
from pygments.lexers import PythonLexer
from pygments.token import Token
from nltk.tokenize import RegexpTokenizer
from io import BytesIO
from pylint.lint import Run

In [2]:
users_code_location_df = pd.read_csv('gender_undersampled_codes (1).csv', index_col=0)

users_code_location_df['code_location'] = users_code_location_df['code_location'].apply(lambda path: 'D:/learning/thesis/' + "/".join(path.split('\\')[3:]))

users_code_location_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Username       1000 non-null   object
 1   Display_Name   1000 non-null   object
 2   Gender         1000 non-null   object
 3   notebook_url   1000 non-null   object
 4   code_location  1000 non-null   object
 5   labels         1000 non-null   object
 6   top_labels     1000 non-null   object
dtypes: object(7)
memory usage: 62.5+ KB


In [3]:
def parse_notebook(notebook_content, notebook_celltype=None):
    code_cells = []
    for cell in notebook_content.cells:
        if not notebook_celltype or cell.cell_type == notebook_celltype:
            if cell.source:
                parsed_code = '\n'.join(line for line in cell.source.split('\n') if not line.strip().startswith('%'))
                code_cells.append(parsed_code)

    return code_cells

In [4]:
def count_tokens(code_sections):

    tokens = set()
    token_pattern = r'[A-Za-z0-9]+'
    tokenizer = RegexpTokenizer(token_pattern)

    for code_section in code_sections :

      try:
        tokens.update(tokenizer.tokenize(code_section))
      except Exception as e:
          print(code_section)
          print("Exception is",e)

    return len(set(tokens))

In [5]:
def char_count(parsed_code, char):
    return parsed_code.count(char)

In [6]:
def saved_words_count(code_sections, saved_word):
  count = 0

  for code_section in code_sections:

    tokens = list(lex(code_section, PythonLexer()))

    count += sum(1 for token_type, token_value in tokens if token_type is Token.Keyword and token_value == saved_word)

  return count

In [7]:
def calculate_comment_density(code_sections):

    total_lines = 0
    comment_lines = 0

    for code_section in code_sections :

      try:
          byte_stream = BytesIO(code_section.encode('utf-8'))

          for token in tokenize.tokenize(byte_stream.readline):
              if token.type == tokenize.NEWLINE or token.type == tokenize.NL:
                  total_lines += 1
              elif token.type == tokenize.COMMENT:
                  comment_lines += 1
      except Exception as e:
          print("Exception is",e)

    single_line_comment_density = comment_lines / total_lines if total_lines else 0

    return single_line_comment_density

In [8]:
def calculate_comment_tokens_density(code_content):
    lines = code_content.split('\n')

    comment_words = 0
    code_words = 0

    for line in lines:
        words = line.split()
        if line.strip().startswith("#"):
            comment_words += len(words)
        else:
            code_words += len(words)

    total_words = comment_words + code_words

    comment_density = comment_words / total_words if total_words > 0 else 0.0

    return comment_density

In [9]:
def extract_naming_from_code(code_sections):

    names_set = set()

    for code_section in code_sections:

      tokens = list(lex(code_section, PythonLexer()))

      names_set.update({token_value for token_type, token_value in tokens if token_type == Token.Name})

    return names_set

In [10]:
import os
import nbformat
import numpy as np

def extract_code_sections(ipynb_file_path):
    print("Starting extraction for:", ipynb_file_path)

    # Check if the file exists
    if not os.path.exists(ipynb_file_path):
        print(f"Error: File not found at {ipynb_file_path}")
        return None

    try:
        # Attempt to read the notebook file
        with open(ipynb_file_path, 'r', encoding='utf-8') as notebook_file:
            notebook_content = nbformat.read(notebook_file, as_version=4)

        # Parse sections
        code_sections = parse_notebook(notebook_content, 'code')
        markdown_sections = parse_notebook(notebook_content, 'markdown')
        all_sections = parse_notebook(notebook_content)

        # Process code sections
        only_code_in_code_sections = [
            " \n ".join([row for row in code_section.split('\n') if not (row.startswith('#') or row.startswith('%'))])
            for code_section in code_sections
        ]
        
        # Calculate features
        number_of_lines = np.sum([code_section.count('\n') + 1 for code_section in code_sections])
        names_set = extract_naming_from_code(code_sections)
        num_of_sections = len(code_sections)
        token_count = count_tokens(code_sections)
        variables_count = len(names_set)
        function_count = saved_words_count(code_sections, "def")
        loop_count = saved_words_count(code_sections, "for")
        condition_count = saved_words_count(code_sections, "if") + saved_words_count(code_sections, "while")

        function_density = function_count / token_count if token_count > 0 else 0
        loop_density = loop_count / token_count if token_count > 0 else 0
        condition_density = condition_count / token_count if token_count > 0 else 0
        single_line_comment_density = calculate_comment_density(code_sections)
        comment_tokens_density = calculate_comment_tokens_density(" \n ".join(code_sections))

        # Compile features
        features = [
            code_sections, markdown_sections, all_sections, only_code_in_code_sections,
            number_of_lines, names_set, num_of_sections, token_count,
            variables_count, function_count, loop_count, condition_count,
            single_line_comment_density,
            function_density, loop_density, condition_density, comment_tokens_density
        ]

        print("Finished extracting features for:", ipynb_file_path)
        return features

    except Exception as e:
        print(f"Error processing file {ipynb_file_path}: {e}")
        return None


In [11]:
users_code_location_df[['code_sections', 'markdown_sections', 'all_sections', 'only_code_in_code_sections',
                'number_of_lines', 'names_set', 'num_of_sections', 'token_count',
                'variables_count', 'function_count', 'loop_count', 'condition_count',
                'single_line_comment_density', 'function_density', 'loop_density',
                'condition_density', 'comment_tokens_density']] = users_code_location_df['code_location'].apply(extract_code_sections).apply(pd.Series)

Starting extraction for: D:/learning/thesis/male/faridrizqis/sql-syntax-using-pandasql-feat-heart-failure-df.ipynb
Finished extracting features for: D:/learning/thesis/male/faridrizqis/sql-syntax-using-pandasql-feat-heart-failure-df.ipynb
Starting extraction for: D:/learning/thesis/male/felipefonte99/cnn-on-images.ipynb
Finished extracting features for: D:/learning/thesis/male/felipefonte99/cnn-on-images.ipynb
Starting extraction for: D:/learning/thesis/male/para24/ovr-vs-multioutput-vs-classifier-chaining.ipynb
Finished extracting features for: D:/learning/thesis/male/para24/ovr-vs-multioutput-vs-classifier-chaining.ipynb
Starting extraction for: D:/learning/thesis/male/ldm314/dnn-keras-santander.ipynb
Finished extracting features for: D:/learning/thesis/male/ldm314/dnn-keras-santander.ipynb
Starting extraction for: D:/learning/thesis/female/tracyporter/play-3-18-tf.ipynb
Finished extracting features for: D:/learning/thesis/female/tracyporter/play-3-18-tf.ipynb
Starting extraction for

C:\ProgramData\anaconda3\Lib\site-packages\nbformat\__init__.py:93: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Finished extracting features for: D:/learning/thesis/male/abhaymudgal/heart-attack-analysis-prediction.ipynb
Starting extraction for: D:/learning/thesis/female/gauri1996/street-market-prediction.ipynb
Finished extracting features for: D:/learning/thesis/female/gauri1996/street-market-prediction.ipynb
Starting extraction for: D:/learning/thesis/male/amarloni/pgs3-e17.ipynb
Finished extracting features for: D:/learning/thesis/male/amarloni/pgs3-e17.ipynb
Starting extraction for: D:/learning/thesis/female/mpwolke/danio-rerio.ipynb
Finished extracting features for: D:/learning/thesis/female/mpwolke/danio-rerio.ipynb
Starting extraction for: D:/learning/thesis/female/parulpandey/melanoma-classification-eda-starter.ipynb
Finished extracting features for: D:/learning/thesis/female/parulpandey/melanoma-classification-eda-starter.ipynb
Starting extraction for: D:/learning/thesis/male/averma111/pss3e18-widedeep-eda.ipynb
Finished extracting features for: D:/learning/thesis/male/averma111/pss3e18

In [12]:
users_code_location_df.head()

,Username,Display_Name,Gender,notebook_url,code_location,labels,top_labels,code_sections,markdown_sections,all_sections,...,token_count,variables_count,function_count,loop_count,condition_count,single_line_comment_density,function_density,loop_density,condition_density,comment_tokens_density
0,faridrizqis,Farid Rizqi S,male,https://www.kaggle.com/code/faridrizqis/sql-sy...,D:/learning/thesis/male/faridrizqis/sql-syntax...,['Heart Failure Prediction Dataset'],{'Heart Failure Prediction Dataset'},"[# Install pandasql\n!pip install -q pandasql,...",[# About Pandasql\npandasql allows you to quer...,[# About Pandasql\npandasql allows you to quer...,...,325,17,0,0,0,0.345178,0.000000,0.000000,0.000000,0.537830
1,felipefonte99,Felipe Loque,male,https://www.kaggle.com/code/felipefonte99/cnn-...,D:/learning/thesis/male/felipefonte99/cnn-on-i...,['LANL Earthquake Prediction'],{'LANL Earthquake Prediction'},[import os\nimport cv2\nimport numpy as np\nim...,[### This kernel was created to the <a target=...,[### This kernel was created to the <a target=...,...,215,113,1,6,3,0.125828,0.004651,0.027907,0.013953,0.090038
2,para24,JP,male,https://www.kaggle.com/code/para24/ovr-vs-mult...,D:/learning/thesis/male/para24/ovr-vs-multiout...,['Mechanisms of Action (MoA) Prediction'],{'Mechanisms of Action (MoA) Prediction'},"[import os, sys\n\nimport numpy as np\nfrom sc...",[* [1. Import Data](#import-data)\n* [2. Data ...,[* [1. Import Data](#import-data)\n* [2. Data ...,...,204,105,3,8,2,0.035211,0.014706,0.039216,0.009804,0.058537
3,ldm314,Brian,male,https://www.kaggle.com/code/ldm314/dnn-keras-s...,D:/learning/thesis/male/ldm314/dnn-keras-santa...,['Santander Customer Transaction Prediction'],{'Santander Customer Transaction Prediction'},[import os\nimport gc\nimport pickle\nfrom pat...,[],[import os\nimport gc\nimport pickle\nfrom pat...,...,445,204,17,6,15,0.140762,0.038202,0.013483,0.033708,0.232595
4,tracyporter,Tracy Porter,female,https://www.kaggle.com/code/tracyporter/play-3...,D:/learning/thesis/female/tracyporter/play-3-1...,['Explore Multi-Label Classification with an E...,{'Explore Multi-Label Classification with an E...,[import numpy as np # linear algebra\nimport p...,[Problem statement\n\nThis is a miltilabel cla...,[Problem statement\n\nThis is a miltilabel cla...,...,191,108,0,6,2,0.107692,0.000000,0.031414,0.010471,0.142857


In [14]:
users_code_location_df.Gender.value_counts()

male      565
female    435
Name: Gender, dtype: int64

In [15]:
users_code_location_df.to_csv('code_sections_extracts.csv', index=False)